In [357]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Suppress Prophet and PLS warnings

def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"\n{model_name} Performance:")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    return {'mse': mse, 'mae': mae}

def fit_prophet_safely(df, periods=1):
    """Safely fit Prophet model with fallback to simple average"""
    try:
        model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            interval_width=0.95,
            mcmc_samples=0
        )
        model.fit(df)
        future = model.make_future_dataframe(periods=periods, freq='2W')
        forecast = model.predict(future)
        return forecast['yhat'].values[-periods:]
    except:
        # Fallback to moving average of last 5 points
        return [df['y'].tail(5).mean()] * periods

print("Loading synthetic data...")
df = pd.read_csv('synthetic_data_confirming.csv', parse_dates=['Time'])

# Prepare data matrices
print("\nPreparing data matrices...")
vocab = df['Topic'].unique()
clusters = df['Cluster'].unique()
dates = df['Time'].unique()
num_dates = len(dates)
test_size = 2  # Number of periods to predict (2 * 2W = 4 недели)

# Create word frequency matrix (dates × words)
word_matrix = pd.pivot_table(
    df,
    values='Count',
    index='Time',
    columns='Topic',
    aggfunc='sum'
).fillna(0)

# Create cluster matrix (dates × clusters)
cluster_matrix = pd.pivot_table(
    df,
    values='Count',
    index='Time',
    columns='Cluster',
    aggfunc='sum'
).fillna(0)

# Split data
X = word_matrix.values               # shape: (num_dates, num_words)
y = cluster_matrix.values            # shape: (num_dates, num_clusters)
X_train = X[:-test_size]             # all but last test_size rows
y_train = y[:-test_size]
X_test = X[-test_size:]
y_test = y[-test_size:]

# Scale features (StandardScaler(with_mean=False) чтобы не вычитать среднее из нулевых значений)
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



Loading synthetic data...

Preparing data matrices...


In [358]:
word_matrix

Topic,Music_core1,Music_core2,Music_core3,Music_core4,Music_noiseSD1,Music_noiseSD2,Music_noiseSD3,Music_noiseSD4,Music_noiseSD5,Music_noiseSD6,...,Tech_noiseSD7,Tech_noiseSD8,Tech_noiseWN1,Tech_noiseWN2,Tech_noiseWN3,Tech_noiseWN4,Tech_noiseWN5,Tech_noiseWN6,Tech_noiseWN7,Tech_noiseWN8
Time,,,,,,,,,,,,,,,,,,,,,
2024-01-07,219.925251,219.750744,220.328193,219.777920,22.191806,21.582285,16.502992,15.592469,9.254181,11.086699,...,9.475893,14.224420,5.739229,0.000000,10.816743,2.156964,11.605758,3.830550,8.955474,3.032632
2024-01-21,246.045238,244.298317,237.402910,227.952463,30.912433,29.624058,22.235561,26.836168,14.066028,5.573662,...,0.000000,29.202366,8.308912,10.370230,7.563166,4.913286,16.750085,11.802621,10.625421,5.931144
2024-02-04,246.734496,251.465553,252.147178,233.820794,13.750701,32.618106,14.521067,11.138891,6.601020,24.535509,...,10.424239,19.285191,5.261615,2.101634,1.122636,8.277705,0.000000,8.561340,0.000000,9.950590
2024-02-18,229.922917,242.075607,262.067718,240.720478,37.310024,42.534413,24.156926,21.930677,0.668612,10.223897,...,13.053693,15.470138,15.170306,5.880758,15.651865,5.038534,7.980195,0.000000,0.423651,0.000000
2024-03-03,230.191607,232.047694,266.673372,248.976006,20.909769,57.796655,21.748700,18.342079,4.928340,13.615837,...,16.233381,23.300853,1.264104,14.673324,0.000000,2.381685,2.208137,8.935405,0.385146,1.051301
2024-03-17,255.405776,238.451193,266.259484,255.799178,45.169216,85.759459,18.115908,21.611554,12.312142,9.718530,...,0.000000,43.414870,6.167645,10.381036,10.514925,4.342341,4.364310,6.168417,0.000000,3.177644
2024-03-31,282.327950,262.867058,263.133153,262.791507,73.151444,361.814698,24.527458,31.670811,7.666057,26.167894,...,5.825667,44.064611,0.545176,3.388816,11.003494,4.469822,20.866060,4.887747,9.976849,0.000000
2024-04-14,282.511557,287.909074,260.458685,270.683343,94.083948,195.407948,27.721456,28.761300,0.000000,27.683324,...,0.000000,56.710276,0.000000,0.000000,0.976279,2.779934,0.000000,2.668889,3.837043,0.084453
2024-04-28,265.877209,294.699619,260.762096,277.697520,362.238354,126.312420,35.474170,33.229776,10.502940,27.785575,...,0.000000,90.960109,1.329385,0.000000,13.354792,3.447224,6.522438,11.744676,3.412911,2.142675


In [359]:
cluster_matrix

Cluster,Music,Sports,Tech
Time,,,
2024-01-07,1038.257486,990.760937,1006.691521
2024-01-21,1152.727232,1068.703916,1117.319567
2024-02-04,1185.456965,1103.387537,1121.746716
2024-02-18,1171.805009,1104.753320,1123.994965
2024-03-03,1181.455021,1083.998076,1142.708973
2024-03-17,1252.903534,1106.497382,1194.267861
2024-03-31,1654.542833,1220.131628,1282.372472
2024-04-14,1515.237887,1254.526993,1285.572016
2024-04-28,1747.934625,1277.247551,1364.818580


In [ ]:
# =============================
# Hypothesis 1: Feature Selection
# =============================

print("\nTesting Hypothesis 1: Feature Selection")
results = []

def fit_pls_safely(X_tr, y_tr, X_te, n_components, return_model=False):
    """
    Fit PLS with numerical-stability fallbacks.
    Если возникает ошибка, уменьшаем n_components. В крайнем случае возвращаем среднее.
    """
    try:
        # Добавляем epsilon, чтобы избежать проблем с нулевыми столбцами
        eps = 1e-10
        X_tr_stable = X_tr + eps
        X_te_stable = X_te + eps

        pls = PLSRegression(n_components=n_components, scale=False)
        pls.fit(X_tr_stable, y_tr)
        if return_model:
            return pls.predict(X_te_stable), pls
        return pls.predict(X_te_stable)
    except:
        # Если не удалось, пробуем с уменьшенным числом компонент
        try:
            n_components_new = max(1, n_components - 1)
            print(f"Retrying PLS with {n_components_new} components...")
            pls = PLSRegression(n_components=n_components_new, scale=False)
            pls.fit(X_tr_stable, y_tr)
            if return_model:
                return pls.predict(X_te_stable), pls
            return pls.predict(X_te_stable)
        except:
            # Крайняя мера: среднее значение по y_train
            print("PLS fallback to mean prediction...")
            mean_pred = np.tile(np.mean(y_tr, axis=0), (X_te.shape[0], 1))
            if return_model:
                return mean_pred, None
            return mean_pred
        
n_components_BASE = 3
n_components = min(n_components_BASE, X_train_scaled.shape[1] - 1)
# Full model на всех признаках
y_pred_full, pls_model = fit_pls_safely(
    X_train_scaled, y_train, X_test_scaled, n_components, return_model=True
)
full_metrics = evaluate_model(y_test, y_pred_full, "Full Model (All Features)")
results.append({"method": "Full Model", **full_metrics})

# Функция для важности признаков по PLS: сумма абсолютных коэффициентов по всем таргетам
# coef_ может быть (n_features, n_targets) ИЛИ (n_targets, n_features)
coef_abs = np.abs(pls_model.coef_)

if coef_abs.shape[0] == X_train_scaled.shape[1]:          # (n_features, n_targets)
    importance = coef_abs.sum(axis=1)                     # → (n_features,)
else:                                                     # (n_targets, n_features)
    importance = coef_abs.sum(axis=0)                     # → (n_features,)


top_k = int(len(vocab) * 0.2)  # 20% от общего числа слов
selected_idx = np.argsort(importance)[::-1][:top_k]

# Выбранные слова (топ-20%)
selected_vocab = [vocab[i] for i in selected_idx]

# Тренируем PLS снова, но только на отобранных признаках
X_train_selected = X_train_scaled[:, selected_idx]
X_test_selected = X_test_scaled[:, selected_idx]
pls_selected = PLSRegression(n_components=min(n_components_BASE, X_train_selected.shape[1] - 1), scale=False)
pls_selected.fit(X_train_selected, y_train)
y_pred_selected = pls_selected.predict(X_test_selected)
selected_metrics = evaluate_model(y_test, y_pred_selected, "Selected Features Model")
results.append({"method": "Selected Features", **selected_metrics})



Testing Hypothesis 1: Feature Selection

Full Model (All Features) Performance:
MSE: 2720.7033
MAE: 43.5407

Selected Features Model Performance:
MSE: 63638.6447
MAE: 214.4504


In [361]:
y_pred_selected

array([[2345.433375  , 2669.94550446, 2496.31183969],
       [2255.8568262 , 2692.24031067, 2313.13965508]])

In [362]:
y_pred_full

array([[2100.47101173, 2156.61331008, 2162.99001799],
       [2143.92224551, 2255.12698081, 2110.3319212 ]])

In [363]:
y_test

array([[2154.55578266, 2161.22627183, 2130.33207428],
       [2149.56039629, 2195.04791629, 2110.78303629]])

In [364]:
importance.shape

(60,)

In [365]:
print(selected_idx)

[ 1 21 41 43  3 23 40  0 20 42  2 22]


In [366]:
len(vocab)

60

In [367]:
selected_vocab

['Sports_core2',
 'Music_core2',
 'Tech_core2',
 'Tech_core4',
 'Sports_core4',
 'Music_core4',
 'Tech_core1',
 'Sports_core1',
 'Music_core1',
 'Tech_core3',
 'Sports_core3',
 'Music_core3']

In [368]:
print(X_train_scaled.shape)

(38, 60)


In [369]:
print(X_train_selected.shape)

(38, 12)


In [370]:
word_predictions.shape

(2, 12)

In [371]:
word_predictions

array([[495.34884106, 494.49512296, 495.27319476, 491.52886394,
        494.47144436, 491.68816132, 519.36416284, 506.79042658,
        518.33382478, 507.87876148, 515.16546203, 506.57446994],
       [515.80892756, 515.14881103, 514.55477592, 501.75456387,
        501.18842212, 500.06732359, 541.56042666, 492.44114782,
        540.87739678, 492.92990988, 535.46607837, 492.14535838]])

In [372]:
cluster_from_words

array([[2016.03287168, 2007.92321625, 2012.95664811],
       [2050.80465635, 2042.82757137, 2050.31091427]])

In [373]:
cluster_from_words

array([[2016.03287168, 2007.92321625, 2012.95664811],
       [2050.80465635, 2042.82757137, 2050.31091427]])

In [374]:

# =============================
# Hypothesis 2: Individual Word Prediction (на отобранных словах)
# =============================
X_train_selected_orig   = X_train[:, selected_idx] 

print("\nTesting Hypothesis 2: Individual Word Prediction")

# --- Метод 1: Direct cluster prediction с Prophet ---
cluster_predictions = np.zeros((test_size, len(clusters)))
for i, cluster in enumerate(clusters):
    cluster_data = pd.DataFrame({
        'ds': dates[:-test_size],
        'y': y_train[:, i]
    })
    cluster_predictions[:, i] = fit_prophet_safely(cluster_data, periods=test_size)

direct_metrics = evaluate_model(y_test, cluster_predictions, "Direct Cluster Prediction")
results.append({"method": "Direct Cluster", **direct_metrics})

# --- Метод 2: Individual word prediction и агрегация по отобранным словам ---
word_predictions = np.zeros((test_size, len(selected_vocab)))

# Используем X_train_selected_orig (отобранные слова) вместо полного X_train
for i, word in enumerate(selected_vocab):
    # Здесь X_train_selected_orig[:, i] соответствует столбцу выбранного слова
    word_data = pd.DataFrame({
        'ds': dates[:-test_size],
        'y': X_train_selected_orig[:, i]
    })
    word_predictions[:, i] = fit_prophet_safely(word_data, periods=test_size)

# Агрегируем прогнозы отобранных слов по кластерам
cluster_from_words = np.zeros((test_size, len(clusters)))
for ci, cluster in enumerate(clusters):
    # Для текущего кластера находим индексы слов в selected_vocab, которые ему принадлежат
    word_indices = [
        idx_sel for idx_sel, w in enumerate(selected_vocab)
        if df[df['Topic'] == w]['Cluster'].iloc[0] == cluster
    ]
    if word_indices:                       # в кластере есть отобранные слова
        print(f"\nCluster: {cluster}")
        for idx_sel in word_indices:
            w = selected_vocab[idx_sel]
            preds = word_predictions[:, idx_sel]
            # preds — массив длиной test_size (у нас 2 периода вперёд)
            print(f"  {w:<25s}  t+1={preds[0]:.2f}   t+2={preds[1]:.2f}")
        # агрегируем
        cluster_from_words[:, ci] = word_predictions[:, word_indices].sum(axis=1)
    else:                                  # ни одного слова не попало
        print(f"\nCluster: {cluster}  —  no selected words")
        cluster_from_words[:, ci] = np.mean(y_train[:, ci])
        
    print(f"\n")
    print(f"  Word-based       t+1={cluster_from_words[0, ci]:.2f}   "
              f"t+2={cluster_from_words[1, ci]:.2f}")
    print(f"  Direct Prophet   t+1={cluster_predictions[0, ci]:.2f}   "
          f"t+2={cluster_predictions[1, ci]:.2f}")
    print(f"  True values       t+1={y_test[0, ci]:.2f}   t+2={y_test[1, ci]:.2f}")

word_based_metrics = evaluate_model(y_test, cluster_from_words, "Word-based Prediction (Selected Words)")
results.append({"method": "Word-based", **word_based_metrics})

# =============================
# Сохраняем результаты и строим график
# =============================

results_df = pd.DataFrame(results)
results_df.to_csv('hypothesis_test_results.csv', index=False)
print("\nResults saved to hypothesis_test_results.csv")

plt.figure(figsize=(12, 6))
plt.bar(results_df['method'], results_df['mse'])
plt.title('MSE Comparison Across Methods')
plt.ylabel('Mean Squared Error')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('hypothesis_comparison.png')
plt.close()
print("Results visualization saved to hypothesis_comparison.png")

# =============================
# Вывод ключевых показателей
# =============================

print("\nKey Findings:")
print("1. Feature Selection Impact:")
feature_improvement = (full_metrics['mse'] - selected_metrics['mse']) / full_metrics['mse'] * 100
print(f"   {'Improvement' if feature_improvement > 0 else 'Degradation'}: {abs(feature_improvement):.1f}%")

print("\n2. Word-level vs Direct Prediction:")
word_improvement = (direct_metrics['mse'] - word_based_metrics['mse']) / direct_metrics['mse'] * 100
print(f"   {'Improvement' if word_improvement > 0 else 'Degradation'}: {abs(word_improvement):.1f}%")


10:22:28 - cmdstanpy - INFO - Chain [1] start processing



Testing Hypothesis 2: Individual Word Prediction


10:22:28 - cmdstanpy - INFO - Chain [1] done processing


10:22:28 - cmdstanpy - INFO - Chain [1] start processing
10:22:28 - cmdstanpy - INFO - Chain [1] done processing
10:22:28 - cmdstanpy - INFO - Chain [1] start processing
10:22:28 - cmdstanpy - INFO - Chain [1] done processing
10:22:29 - cmdstanpy - INFO - Chain [1] start processing
10:22:29 - cmdstanpy - INFO - Chain [1] done processing
10:22:29 - cmdstanpy - INFO - Chain [1] start processing



Direct Cluster Prediction Performance:
MSE: 353471.1341
MAE: 586.7020


10:22:29 - cmdstanpy - INFO - Chain [1] done processing
10:22:29 - cmdstanpy - INFO - Chain [1] start processing
10:22:29 - cmdstanpy - INFO - Chain [1] done processing
10:22:29 - cmdstanpy - INFO - Chain [1] start processing
10:22:42 - cmdstanpy - INFO - Chain [1] done processing
10:22:43 - cmdstanpy - INFO - Chain [1] start processing
10:22:43 - cmdstanpy - INFO - Chain [1] done processing
10:22:43 - cmdstanpy - INFO - Chain [1] start processing
10:22:44 - cmdstanpy - INFO - Chain [1] start processing
10:22:44 - cmdstanpy - INFO - Chain [1] done processing
10:22:44 - cmdstanpy - INFO - Chain [1] start processing
10:22:44 - cmdstanpy - INFO - Chain [1] done processing
10:22:44 - cmdstanpy - INFO - Chain [1] start processing
10:22:44 - cmdstanpy - INFO - Chain [1] done processing
10:22:44 - cmdstanpy - INFO - Chain [1] start processing
10:22:44 - cmdstanpy - INFO - Chain [1] done processing
10:22:44 - cmdstanpy - INFO - Chain [1] start processing
10:22:44 - cmdstanpy - INFO - Chain [1]


Cluster: Sports
  Sports_core2               t+1=495.35   t+2=515.81
  Sports_core4               t+1=494.47   t+2=501.19
  Sports_core1               t+1=507.88   t+2=492.93
  Sports_core3               t+1=518.33   t+2=540.88


  Word-based       t+1=2016.03   t+2=2050.80
  Direct Prophet   t+1=2577.57   t+2=2696.56
  True values       t+1=2154.56   t+2=2149.56

Cluster: Music
  Music_core2                t+1=494.50   t+2=515.15
  Music_core4                t+1=472.17   t+2=472.17
  Music_core1                t+1=506.57   t+2=492.15
  Music_core3                t+1=515.17   t+2=535.47


  Word-based       t+1=1988.41   t+2=2014.93
  Direct Prophet   t+1=2778.96   t+2=2822.26
  True values       t+1=2161.23   t+2=2195.05

Cluster: Tech
  Tech_core2                 t+1=495.27   t+2=514.55
  Tech_core4                 t+1=491.53   t+2=501.75
  Tech_core1                 t+1=506.79   t+2=492.44
  Tech_core3                 t+1=519.36   t+2=541.56


  Word-based       t+1=2012.96   t+2=2